# Détection d'anomalies routières — Étape 3b : Modèle Cityscapes + score d'incertitude

**Notebook :** `03b_cityscapes_incertitude.ipynb`

Les modèles de l'étape 3a ont **appris** ce qu'est une anomalie à partir de nos masques. Cette
approche est différente : on prend un réseau de segmentation **déjà entraîné sur Cityscapes**
(scènes urbaines, 19 classes « normales » : route, trottoir, voiture, piéton…) et on ne l'entraîne
pas du tout. Il n'a jamais vu d'anomalie. L'idée :

> Un modèle qui ne connaît que les objets « normaux » d'une route **hésite** quand il rencontre un
> objet qu'il ne connaît pas (un cheval, un rocher). On mesure cette hésitation pixel par pixel :
> c'est le **score d'anomalie**.

C'est la détection d'anomalies **hors distribution** (*out-of-distribution*, OOD). Elle
correspond exactement à la définition d'anomalie du dataset (étape 1 : est anomal ce qui ne fait
pas partie des classes de Cityscapes) et c'est l'usage prévu de RoadAnomaly par ses auteurs.

| Ce qui change par rapport à 03a | 03a (A, B, C) | 03b |
|---|---|---|
| Masques d'anomalie utilisés pour apprendre | oui (≈ 36 images par tour) | **aucun** |
| Entraînement | 15 entraînements | **aucun** (inférence seulement) |
| Peut détecter un type d'anomalie jamais vu | seulement s'il ressemble à un exemple d'entraînement | oui, en principe |
| Rôle des masques | entraînement, seuil, évaluation | seuil (validation) et évaluation uniquement |

**GPU :** recommandé (*Exécution → Modifier le type d'exécution → GPU T4*) ; l'inférence prend
environ 1 minute sur GPU, 10 à 15 minutes sur CPU.

| § | Section |
|---|---|
| 0 | Chargement |
| 1 | Le modèle SegFormer entraîné sur Cityscapes |
| 2 | Trois scores d'incertitude |
| 3 | Calcul des scores sur les 60 images |
| 4 | Ce que « voit » le modèle sur les anomalies |
| 5 | Évaluation : AUROC, AP, FPR95 et validation croisée |
| 6 | Exemples visuels |
| 7 | Sauvegarde et lecture des résultats |

## 0. Chargement

Sélectionnez **en une seule fois** `RoadAnomaly.zip` et `roadanomaly_lib.py` (de
`resultats_etape2.zip`), comme pour 03a.

In [ ]:
import os, sys, zipfile, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
BASE = Path("/content") if IN_COLAB else Path.cwd()
RESULTS = BASE / "resultats_etape3b"
for d in ["preds", "figures"]:
    (RESULTS / d).mkdir(parents=True, exist_ok=True)

def trouver_dataset(dossier):
    dossier = Path(dossier)
    hits = sorted(dossier.rglob("frame_list.json")) if dossier.exists() else []
    return hits[0].parent if hits else None

if IN_COLAB:
    os.chdir(BASE)
    if trouver_dataset(BASE / "data") is None or not (BASE / "roadanomaly_lib.py").exists():
        from google.colab import files
        print("Sélectionnez RoadAnomaly.zip et roadanomaly_lib.py :")
        for nom in files.upload():
            if nom.startswith("RoadAnomaly") and nom.endswith(".zip"):
                with zipfile.ZipFile(BASE / nom) as z: z.extractall(BASE / "data")
                os.remove(BASE / nom)
            elif nom.startswith("roadanomaly_lib") and nom.endswith(".py") and nom != "roadanomaly_lib.py":
                shutil.move(BASE / nom, BASE / "roadanomaly_lib.py")
    DATA_ROOT = trouver_dataset(BASE / "data")
else:
    DATA_ROOT = next((p for p in [Path("RoadAnomaly"), Path("../RoadAnomaly")]
                      if (p / "frame_list.json").exists()), None)

assert DATA_ROOT is not None, "Dataset introuvable : envoyez RoadAnomaly.zip"
assert (BASE / "roadanomaly_lib.py").exists(), "roadanomaly_lib.py manquant (dans resultats_etape2.zip)"

import importlib, json, time, warnings
import numpy as np, pandas as pd, torch, torch.nn.functional as F
import matplotlib.pyplot as plt, cv2
try:
    import albumentations
    assert int(albumentations.__version__.split(".")[0]) >= 2
except (ImportError, AssertionError):
    %pip install -q "albumentations==2.0.8"
    importlib.invalidate_caches()
sys.path.insert(0, str(BASE))
import roadanomaly_lib; importlib.reload(roadanomaly_lib)
from roadanomaly_lib import *

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
splits = make_splits(DATA_ROOT)
frames = splits["name"].tolist()
print("Dataset :", DATA_ROOT, "| appareil :", device, "| images :", len(frames))

plt.rcParams.update({"figure.dpi": 80, "savefig.dpi": 150, "savefig.bbox": "tight",
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e4e3df", "axes.axisbelow": True})

## 1. Le modèle : SegFormer entraîné sur Cityscapes

**SegFormer** (Xie et al., NVIDIA, 2021) est un réseau de segmentation à base de
**Transformers** : l'encodeur découpe l'image en petits morceaux (*patches*) et les met en
relation grâce au mécanisme d'**attention**, ce qui lui donne une vision globale de la scène.
Son décodeur est un simple réseau de neurones multicouche (MLP).

On utilise la version **B5** (85 millions de paramètres, la plus précise) **entraînée sur
Cityscapes** et publiée par NVIDIA sur Hugging Face. Elle prédit, pour chaque pixel, un score
(*logit*) pour chacune des 19 classes :

`road, sidewalk, building, wall, fence, pole, traffic light, traffic sign, vegetation, terrain,
sky, person, rider, car, truck, bus, train, motorcycle, bicycle`

Aucune de ces classes n'est un animal, un rocher, un cône ou un pneu : c'est précisément ce qui
permet de détecter ces objets comme « inconnus ».

**Pourquoi SegFormer plutôt qu'un DeepLab entraîné sur Cityscapes ?** Les bibliothèques
standard (torchvision, smp) ne fournissent pas de poids Cityscapes ; SegFormer est le modèle
Cityscapes de référence le plus simple à charger (une ligne avec `transformers`, préinstallée
sur Colab). Le téléchargement des poids (≈ 340 Mo) prend environ 30 s.

In [ ]:
from transformers import SegformerForSemanticSegmentation, SegformerConfig

CITYSCAPES = ["road", "sidewalk", "building", "wall", "fence", "pole", "traffic light", "traffic sign",
              "vegetation", "terrain", "sky", "person", "rider", "car", "truck", "bus", "train",
              "motorcycle", "bicycle"]
CANDIDATES = ["nvidia/segformer-b5-finetuned-cityscapes-1024-1024",   # le plus précis
              "nvidia/segformer-b2-finetuned-cityscapes-1024-1024"]   # repli plus léger

if os.environ.get("RA_NO_PRETRAINED"):        # tests hors ligne uniquement : poids aléatoires
    seg = SegformerForSemanticSegmentation(SegformerConfig(num_labels=19)); MODEL_ID = "test-aléatoire"
else:
    for MODEL_ID in CANDIDATES:
        try:
            seg = SegformerForSemanticSegmentation.from_pretrained(MODEL_ID); break
        except Exception as e:
            print(f"{MODEL_ID} indisponible ({type(e).__name__}), essai suivant…")
    else:
        raise RuntimeError("Aucun modèle SegFormer Cityscapes n'a pu être téléchargé")
seg = seg.to(device).eval()
labels = [seg.config.id2label[i] for i in range(19)] if seg.config.id2label.get(0, "") == "road" else CITYSCAPES
print(f"Modèle : {MODEL_ID} | {sum(p.numel() for p in seg.parameters()) / 1e6:.0f} M paramètres")
print("Classes :", ", ".join(labels))

## 2. Trois scores d'incertitude

Pour un pixel, le modèle donne 19 logits *z₁ … z₁₉*. La fonction **softmax** les transforme en
probabilités *p₁ … p₁₉* (positives, de somme 1). On compare trois façons classiques de mesurer
l'hésitation :

| Score | Formule | Idée | Référence |
|---|---|---|---|
| **MSP** (*Maximum Softmax Probability*) | 1 − max *pᵢ* | si la classe la plus probable ne l'est qu'à 40 %, le modèle hésite | Hendrycks & Gimpel, 2017 (la référence de base) |
| **Max-logit** | − max *zᵢ* | le logit brut, avant softmax, garde l'information « aucune classe ne correspond vraiment » que le softmax efface en normalisant | Hendrycks et al., 2022 ; souvent meilleur sur les scènes routières |
| **Entropie** | − Σ *pᵢ* log *pᵢ* / log 19 | mesure la dispersion de toute la distribution, pas seulement la classe maximale | — |

**Limite connue :** ces scores sont élevés **aux frontières entre objets** (bord route/trottoir,
contours des voitures), où le modèle hésite légitimement entre deux classes connues. Ce sont des
fausses alarmes typiques de cette approche ; on les verra dans les exemples (section 6).

**Mise à l'échelle.** Les trois scores sont ramenés entre 0 et 1, puis stockés sur 256 niveaux
comme les probabilités de 03a, pour que l'étape 4 compare tous les modèles de la même façon.
MSP et entropie sont déjà dans [0, 1]. Le max-logit est ramené à [0, 1] par une transformation
**fixe et croissante**, calculée sur les scores de toutes les images **sans regarder les
masques** : elle ne change aucun classement de pixels, donc aucune AUROC ni AP.

In [ ]:
MEAN_T = torch.tensor(MEAN, device=device).view(1, 3, 1, 1)
STD_T = torch.tensor(STD, device=device).view(1, 3, 1, 1)

@torch.no_grad()
def scores_image(img):
    # img : RGB uint8 1280x720 -> (msp, maxlogit brut, entropie, classe prédite) à 960x540
    x = torch.from_numpy(img).to(device).permute(2, 0, 1)[None].float() / 255
    x = (x - MEAN_T) / STD_T                                   # même normalisation que SegFormer
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
        logits = seg(pixel_values=x).logits                     # (1, 19, 180, 320) : résolution 1/4
    logits = F.interpolate(logits.float(), size=(WORK_H, WORK_W), mode="bilinear", align_corners=False)[0]
    p = logits.softmax(0)
    msp = 1 - p.max(0).values
    maxlogit = -logits.max(0).values
    entropy = -(p * torch.log(p.clamp_min(1e-12))).sum(0) / np.log(19)
    return (msp.cpu().numpy(), maxlogit.cpu().numpy(), entropy.cpu().numpy(),
            logits.argmax(0).to(torch.uint8).cpu().numpy())

## 3. Calcul des scores sur les 60 images

Les images sont données **en pleine résolution** (1280 × 720) : SegFormer a été entraîné sur
des découpes de 1024 × 1024 d'images Cityscapes 2048 × 1024, une échelle proche. Les scores sont
ensuite ramenés à 960 × 540, la résolution de travail de tout le projet.

In [ ]:
t0 = time.time()
MSP, MAXLOGIT, ENT, CLS, MASKS = {}, {}, {}, {}, {}
for i, n in enumerate(frames):
    a, b, c, d = scores_image(load_image(DATA_ROOT, n))
    MSP[n], MAXLOGIT[n], ENT[n], CLS[n] = a, b.astype(np.float32), c, d
    MASKS[n] = cv2.resize(load_mask(DATA_ROOT, n), (WORK_W, WORK_H), interpolation=cv2.INTER_NEAREST).astype(bool)
    if (i + 1) % 10 == 0:
        print(f"{i + 1}/{len(frames)} images ({time.time() - t0:.0f} s)")

# max-logit -> [0, 1] : transformation fixe et croissante, calculée sans les masques
tous = np.concatenate([v.ravel()[::97] for v in MAXLOGIT.values()])
lo, hi = np.percentile(tous, [0.5, 99.5])
print(f"max-logit brut : 0,5 % = {lo:.2f}, 99,5 % = {hi:.2f} (bornes de la mise à l'échelle)")

def to_u8(s):
    return (np.clip(s, 0, 1) * 255).round().astype(np.uint8)

SCORES = {
    "D_ood_msp":      {n: to_u8(MSP[n] / (1 - 1 / 19)) for n in frames},   # max théorique de 1 - max p
    "D_ood_maxlogit": {n: to_u8((MAXLOGIT[n] - lo) / (hi - lo)) for n in frames},
    "D_ood_entropy":  {n: to_u8(ENT[n]) for n in frames},
}
LABELS = {"D_ood_msp": "D1. Cityscapes + MSP", "D_ood_maxlogit": "D2. Cityscapes + max-logit",
          "D_ood_entropy": "D3. Cityscapes + entropie"}
COLORS = {"D_ood_msp": "#8a8984", "D_ood_maxlogit": "#2a78d6", "D_ood_entropy": "#1baf7a"}
del MAXLOGIT

## 4. Ce que « voit » le modèle sur les anomalies

Avant de mesurer les scores, regardons **quelle classe Cityscapes** le modèle attribue aux pixels
anomaux. Cela explique à l'avance ce qu'il détectera bien ou mal :

- un pixel anomal attribué **avec hésitation** à une classe quelconque → bien détecté ;
- un pixel anomal attribué **avec assurance** à une classe proche (une vache → *terrain*, un
  rocher → *road*) → anomalie manquée, car le modèle ne doute pas.

In [ ]:
rows = []
for n in frames:
    cat = category_of(n); m = MASKS[n]
    cnt = np.bincount(CLS[n][m], minlength=19)
    for c in range(19):
        if cnt[c]:
            rows.append(dict(category=cat, classe=labels[c], pixels=cnt[c]))
cls_df = pd.DataFrame(rows)
part = cls_df.pivot_table(index="category", columns="classe", values="pixels", aggfunc="sum", fill_value=0)
part = (part.T / part.sum(axis=1)).T.reindex(CATEGORIES)
top = part.sum().sort_values(ascending=False).index[:8]
tab = (part[top] * 100).round(0).astype(int)
tab["autres"] = 100 - tab.sum(axis=1)
tab.index = [{"animals": "animaux", "obstacles": "obstacles", "cones": "cônes",
              "lost_tires": "pneus perdus", "vehicle": "véhicules"}[c] for c in tab.index]
print("Classe Cityscapes prédite pour les pixels ANOMAUX (% des pixels anomaux de chaque catégorie)")
display(tab)

**Comment lire ce tableau :** chaque ligne montre en quoi le modèle « transforme » une catégorie
d'anomalie. Par exemple, si les animaux sont surtout vus comme *person* ou *terrain*, et les
rochers comme *road* ou *terrain*, ce sont des confusions plausibles qui rendent ces anomalies
plus difficiles à détecter.

## 5. Évaluation

### 5.1 Métriques indépendantes du seuil, sur les 60 images

Comme le modèle n'a **rien appris** de nos masques, on peut l'évaluer directement sur les 60
images, sans fuite de données. Ce sont les trois métriques standard du benchmark RoadAnomaly :

| Métrique | Signification | Meilleur |
|---|---|---|
| **AUROC** | probabilité qu'un pixel anomal tiré au hasard ait un score plus élevé qu'un pixel de fond tiré au hasard | 1 (0,5 = hasard) |
| **AP** (*Average Precision* ≈ aire sous la courbe précision-rappel, AUPRC) | qualité de la détection en tenant compte de la rareté des anomalies ; beaucoup plus sévère que l'AUROC quand la classe positive est rare | 1 (hasard ≈ 0,10 ici) |
| **FPR95** | part des pixels de fond signalés à tort quand on règle le seuil pour retrouver 95 % des pixels anomaux | 0 |

In [ ]:
def hists(score_dict, names):
    pos, neg = np.zeros(256, np.int64), np.zeros(256, np.int64)
    for n in names:
        s, m = score_dict[n], MASKS[n]
        pos += np.bincount(s[m], minlength=256); neg += np.bincount(s[~m], minlength=256)
    return pos, neg

def curve_metrics(pos, neg):
    tp = pos[::-1].cumsum(); fp = neg[::-1].cumsum()          # seuils décroissants 255 -> 0
    tpr = np.r_[0, tp / pos.sum()]; fpr = np.r_[0, fp / neg.sum()]
    auroc = np.trapezoid(tpr, fpr) if hasattr(np, "trapezoid") else np.trapz(tpr, fpr)
    prec = tp / np.maximum(tp + fp, 1)
    ap = np.sum(np.diff(np.r_[0, tp / pos.sum()]) * prec)
    fpr95 = fpr[np.argmax(tpr >= 0.95)]
    return dict(AUROC=auroc, AP=ap, FPR95=fpr95)

def metrics_at(pos, neg, t):
    tp, fn, fp = pos[t:].sum(), pos[:t].sum(), neg[t:].sum()
    return dict(precision=tp / max(tp + fp, 1), recall=tp / max(tp + fn, 1),
                f1=2 * tp / max(2 * tp + fp + fn, 1), iou=tp / max(tp + fp + fn, 1))

def best_threshold(pos, neg):
    tp = pos[::-1].cumsum()[::-1]; fp = neg[::-1].cumsum()[::-1]; fn = pos.sum() - tp
    f1 = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
    return int(np.argmax(f1[1:]) + 1)

glob = []
for key, sc in SCORES.items():
    pos, neg = hists(sc, frames)
    glob.append(dict(score=LABELS[key], **curve_metrics(pos, neg)))
glob = pd.DataFrame(glob).set_index("score")
print(f"Part de pixels anomaux (AP d'un détecteur au hasard) : {np.mean([m.mean() for m in MASKS.values()]):.3f}")
display(glob.style.format("{:.3f}"))

### 5.2 Mêmes conditions que 03a : seuil sur la validation, scores sur le test

Pour comparer équitablement avec les modèles A, B, C, on reproduit **exactement** leur protocole :
pour chaque tour *k*, le seuil est choisi sur le pli de validation (*k* + 1), puis l'IoU et le F1
sont mesurés sur le pli de test *k*. Les fichiers sont sauvegardés au même format que 03a.

In [ ]:
summ = []
for key, sc in SCORES.items():
    for k in range(N_FOLDS):
        _, val, test = fold_sets(splits, k)
        t = best_threshold(*hists(sc, val))
        vm, tm = metrics_at(*hists(sc, val), t), metrics_at(*hists(sc, test), t)
        np.savez_compressed(RESULTS / "preds" / f"{key}_fold{k}.npz", **{n: sc[n] for n in test})
        s = dict(model=key, fold=k, best_epoch=0, epochs_run=0, threshold=t / 255,
                 val_iou=vm["iou"], val_f1=vm["f1"], test_iou=tm["iou"], test_f1=tm["f1"],
                 test_precision=tm["precision"], test_recall=tm["recall"], minutes=0.0)
        json.dump(s, open(RESULTS / f"summary_{key}_fold{k}.json", "w"), indent=1)
        summ.append(s)
summ = pd.DataFrame(summ)
summ["score"] = summ.model.map(LABELS)

par_tour = summ.pivot_table(index="score", columns="fold", values="test_iou").round(3)
par_tour.columns = [f"tour {c}" for c in par_tour.columns]
display(par_tour)

agg = summ.groupby("score")[["test_iou", "test_f1", "test_precision", "test_recall", "threshold"]].agg(["mean", "std"])
display(pd.DataFrame(index=agg.index, data={
    "IoU test": [f"{m:.3f} ± {s:.3f}" for m, s in zip(agg.test_iou["mean"], agg.test_iou["std"])],
    "F1 (= Dice) test": [f"{m:.3f} ± {s:.3f}" for m, s in zip(agg.test_f1["mean"], agg.test_f1["std"])],
    "précision": agg.test_precision["mean"].map("{:.3f}".format),
    "rappel": agg.test_recall["mean"].map("{:.3f}".format),
    "seuil moyen": agg.threshold["mean"].map("{:.2f}".format)}))
summ.to_csv(RESULTS / "resume_cityscapes_ood.csv", index=False)
glob.to_csv(RESULTS / "metriques_globales_ood.csv")

In [ ]:
# Courbes précision-rappel des trois scores (60 images)
fig, ax = plt.subplots(figsize=(6.5, 4.6))
for key, sc in SCORES.items():
    pos, neg = hists(sc, frames)
    tp = pos[::-1].cumsum(); fp = neg[::-1].cumsum()
    rec, prec = tp / pos.sum(), tp / np.maximum(tp + fp, 1)
    ax.plot(rec, prec, color=COLORS[key], lw=2,
            label=f"{LABELS[key]}  (AP {glob.loc[LABELS[key], 'AP']:.3f})")
base = np.mean([m.mean() for m in MASKS.values()])
ax.axhline(base, color="#8a8984", ls="--", lw=1)
ax.text(0.02, base + 0.015, f"hasard ({base:.2f})", fontsize=9, color="#52514e")
ax.set_xlabel("Rappel"); ax.set_ylabel("Précision"); ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
ax.set_title("Courbes précision-rappel, approche Cityscapes (60 images)", fontweight="bold")
ax.legend(frameon=False, fontsize=9, loc="upper right")
plt.tight_layout(); fig.savefig(RESULTS / "figures" / "01_courbes_precision_rappel.png"); plt.show()

## 6. Exemples visuels

Pour chaque image : la vérité terrain, la **segmentation Cityscapes** (ce que le modèle croit
voir), puis les trois cartes de score (blanc = score d'anomalie élevé). Contour cyan = vérité
terrain.

In [ ]:
PALETTE = np.array([[128, 64, 128], [244, 35, 232], [70, 70, 70], [102, 102, 156], [190, 153, 153],
                    [153, 153, 153], [250, 170, 30], [220, 220, 0], [107, 142, 35], [152, 251, 152],
                    [70, 130, 180], [220, 20, 60], [255, 0, 0], [0, 0, 142], [0, 0, 70], [0, 60, 100],
                    [0, 80, 100], [0, 0, 230], [119, 11, 32]], np.uint8)   # couleurs officielles Cityscapes

choix = ["animals22_red_fox_on_road", "obstacles09_boulder3", "cones03_Bang_Na_Trat_Highway",
         "lost_tires08", "animals24_Sheep_Herders_Block_the_Road_Kaleybar_to_Eskanlu"]
choix = [n for n in choix if n in MASKS]
cols = ["Image", "Segmentation Cityscapes"] + [LABELS[k] for k in SCORES]
fig, axes = plt.subplots(len(choix), len(cols), figsize=(3.2 * len(cols), 1.95 * len(choix)))
for r, n in enumerate(choix):
    img = cv2.resize(load_image(DATA_ROOT, n), (WORK_W, WORK_H), interpolation=cv2.INTER_AREA)
    cont, _ = cv2.findContours(MASKS[n].astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    panels = [img, PALETTE[CLS[n]]] + [cv2.cvtColor(SCORES[k][n], cv2.COLOR_GRAY2RGB) for k in SCORES]
    for c, p in enumerate(panels):
        axes[r, c].imshow(cv2.drawContours(p.copy(), cont, -1, (0, 200, 255), 2) if c else p)
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([]); axes[r, c].grid(False)
        if r == 0: axes[r, c].set_title(cols[c], fontsize=9)
    axes[r, 0].set_ylabel(n[:20], fontsize=8)
plt.tight_layout(); fig.savefig(RESULTS / "figures" / "02_exemples_cartes_ood.png"); plt.show()

**Ce qu'il faut regarder sur ces cartes :**

- les **anomalies** apparaissent-elles plus claires que la route autour ?
- les **frontières entre objets connus** (bords de route, contours des voitures, piétons) sont-elles
  aussi claires ? Ce sont les fausses alarmes typiques de l'approche ;
- la **segmentation Cityscapes** : en quelle classe le modèle transforme-t-il l'anomalie ? Une
  classe affichée nettement et uniformément signifie que le modèle est sûr de lui, donc que
  l'anomalie sera manquée.

## 7. Sauvegarde et lecture des résultats

Les fichiers `preds/D_ood_*_fold<k>.npz` et `summary_*.json` ont le même format que ceux de 03a :
l'étape 4 (benchmark) chargera les deux archives et comparera tous les modèles avec exactement
les mêmes métriques.

**Comment interpréter vos chiffres (à comparer au tableau de 03a) :**

1. **Le meilleur des trois scores.** Le max-logit dépasse souvent MSP sur les scènes routières,
   car le softmax « écrase » l'information utile en normalisant. Vérifiez-le sur vos AP.
2. **Supervisé contre non supervisé.** On s'attend à ce que l'approche Cityscapes ait une IoU
   **nettement plus faible** que B et C (0,64–0,66), qui ont appris nos anomalies. Ce n'est pas un
   échec : elle n'a vu **aucune** anomalie, et elle détecterait un objet d'un type totalement
   nouveau, alors que B et C ne reconnaissent que ce qui ressemble à leurs exemples d'entraînement.
3. **Le coût caché du supervisé.** B et C ont été entraînés et testés sur des images du même
   dataset (mêmes types d'anomalies, mêmes sources d'images). Sur un nouveau type d'objet, leur
   score pourrait chuter fortement ; l'étape 4 l'examinera avec le tour 3 (pneus jamais vus).
4. **Domaine.** SegFormer a été entraîné sur des rues allemandes (Cityscapes). Les routes de
   RoadAnomaly (pistes, campagne, Afrique, Asie) sont très différentes : une partie des fausses
   alarmes vient de scènes « inconnues » dans leur ensemble, pas seulement des objets anomaux.

In [ ]:
shutil.make_archive(str(BASE / "resultats_etape3b"), "zip", RESULTS)
if IN_COLAB:
    from google.colab import files
    files.download(str(BASE / "resultats_etape3b.zip"))
print(sorted(str(p.relative_to(RESULTS)) for p in RESULTS.rglob("*") if p.is_file())[:8], "…")